# Event handlers

`next_action()` pauses a cell until a click arrives. This notebook uses callbacks instead, so the kernel stays free while the surface is live.

`on_action` runs for each action event, and `on_message` runs for every renderer-to-agent message. Each registration returns an `unsubscribe` function.

In [ ]:
import sys
from importlib.util import find_spec

# Pyodide and xeus-python both report sys.platform "emscripten"; only Pyodide ships the pyodide package.
if sys.platform != "emscripten":
    KERNEL = "cpython"
else:
    try:
        has_pyodide = find_spec("pyodide") is not None
    except ValueError:  # xeus-python's pyodide stub has __spec__ set to None
        has_pyodide = False
    KERNEL = "pyodide" if has_pyodide else "xpython"

if KERNEL == "pyodide" and find_spec("a2ui_jupyter") is None:
    import piplite

    await piplite.install(["rpds-py", "a2ui-jupyter"])

from a2ui_jupyter import A2UISurface

BASIC_CATALOG_ID = "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"


def new_surface(surface_id, *, send_data_model=False, **widget_kwargs):
    surface = A2UISurface(**widget_kwargs)
    create = {"surfaceId": surface_id, "catalogId": BASIC_CATALOG_ID}
    if send_data_model:
        create["sendDataModel"] = True
    surface.send({"version": "1.0", "createSurface": create})
    return surface

In [ ]:
SURFACE_ID = "event-handlers"
surface = new_surface(SURFACE_ID)
count = 0


def on_increment(message):
    global count
    payload = message.model_dump(by_alias=True, exclude_unset=True)["action"]
    if payload["name"] != "increment":
        return
    count += payload["context"]["step"]
    surface.send(
        {
            "version": "1.0",
            "updateDataModel": {"surfaceId": SURFACE_ID, "path": "/count", "value": count},
        }
    )


def on_any_message(message):
    payload = message.model_dump(by_alias=True, exclude_unset=True)
    kind = next(key for key in payload if key != "version")
    surface.send(
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": SURFACE_ID,
                "path": "/log",
                "value": f"Last renderer message: {kind}",
            },
        }
    )


unsubscribe_action = surface.on_action(on_increment)
unsubscribe_message = surface.on_message(on_any_message)

surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": SURFACE_ID,
                "value": {"count": 0, "log": "No messages yet"},
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": SURFACE_ID,
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["title", "count_text", "add_button", "log_text"],
                        "align": "center",
                    },
                    {"id": "title", "component": "Text", "text": "## Counter"},
                    {
                        "id": "count_text",
                        "component": "Text",
                        "text": {"@path": "/count"},
                    },
                    {
                        "id": "add_button",
                        "component": "Button",
                        "child": "add_label",
                        "variant": "primary",
                        "action": {
                            "event": {
                                "name": "increment",
                                "context": {"step": 2},
                            }
                        },
                    },
                    {"id": "add_label", "component": "Text", "text": "Add 2"},
                    {
                        "id": "log_text",
                        "component": "Text",
                        "text": {"@path": "/log"},
                        "variant": "caption",
                    },
                ],
            },
        },
    ]
)
surface

Click **Add 2** a few times. The count updates in the surface, and the log line shows the last message type. This cell is already finished, so you can run other cells while you click.

Run this cell to remove the counter handler. Clicks still reach `on_any_message`, so the log keeps updating, but the count stays the same.

In [ ]:
unsubscribe_action()
print("Counter handler removed. The message handler is still active.")

Run this cell to remove the message handler as well. The surface stays on screen but no longer changes in response to clicks.

In [ ]:
unsubscribe_message()
print("All handlers removed.")